In [87]:
import pandas as pd
from yaml import safe_dump, safe_load

from rfm_sp500 import constants, validation

with open(constants.CONFIG_FILE, "r") as f:
    config = safe_load(f)

start_date = config["start_date"]
end_date_exclusive = config["end_date_exclusive"]

In [88]:
sp_dir = constants.RAW_DATA_DIR / f"sp500_data_{start_date}_{end_date_exclusive}.csv.gz"
sp500_df = pd.read_csv(sp_dir, parse_dates=["Date"])
empty_tickers = ["FDXF", "HONA", "Q", "SNDK", "VYLR"]
sp500_df = sp500_df[~sp500_df["Ticker"].isin(empty_tickers)]
sp500_df.head()

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009102,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-05,A,114.941277,119.610001,0.000,120.320000,118.070000,118.480003,0.0,2344000.0
2,2021-01-06,A,118.093231,122.889999,0.000,123.879997,118.209999,118.690002,0.0,2428500.0
3,2021-01-07,A,121.235580,126.160004,0.000,127.480003,124.110001,124.330002,0.0,1775900.0
4,2021-01-08,A,122.100464,127.059998,0.000,127.639999,125.849998,126.489998,0.0,1577200.0


In [89]:
# Check for duplicates

duplicate_ticker_date = sp500_df[sp500_df.duplicated(subset=["Ticker", "Date"], keep=False)]
duplicate_ticker_date

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


No duplicates

In [90]:
def flatten(x):
    summary, middle = validation.get_nan_summary(x)
    return pd.Series({**summary, **{f"middle_{col}": n for col, n in middle.items()}})


nan_summary_df = sp500_df.groupby("Ticker").apply(flatten)
nan_summary_df = nan_summary_df.loc[~nan_summary_df.eq(0).all(axis=1)].reset_index()

nan_save_dir = constants.PROCESSED_DATA_DIR / "nan_summary_sp500.csv.gz"

nan_summary_df.to_csv(nan_save_dir, index=False)
print(f"Saved S&P 500 NaN summary to {nan_save_dir}")

nan_summary_df = nan_summary_df.replace(0, pd.NA)
nan_summary_df = nan_summary_df.dropna(axis=1)
nan_summary_df

Saved S&P 500 NaN summary to D:\dev\crm-rfm-sp500\data\processed\nan_summary_sp500.csv.gz


,Ticker,Leading
0,APP,70
1,CEG,263
2,COIN,69
3,EXE,26
4,GEHC,492
5,GEV,812
6,HOOD,143
7,KVUE,587
8,RDDT,808
9,SOLV,811


There is only data missing leading data meaning these 11 stocks were added after the start date and there is no trailing missing data or data missing in the middle

Therefore, can focus on deciding how to handle unequal history from leading values only

In [91]:
leading_tickers = nan_summary_df["Ticker"].unique()

In [92]:
benchmark_dir = constants.RAW_DATA_DIR / f"benchmark_data_{start_date}_{end_date_exclusive}.csv.gz"
benchmark_df = pd.read_csv(benchmark_dir, parse_dates=["Date"])
benchmark_df.head()

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,3700.649902,3700.649902,0.0,3769.989990,3662.709961,3764.610107,0.0,5015000000
1,2021-01-05,3726.860107,3726.860107,0.0,3737.830078,3695.070068,3698.020020,0.0,4591020000
2,2021-01-06,3748.139893,3748.139893,0.0,3783.040039,3705.340088,3712.199951,0.0,6064110000
3,2021-01-07,3803.790039,3803.790039,0.0,3811.550049,3764.709961,3764.709961,0.0,5099160000
4,2021-01-08,3824.679932,3824.679932,0.0,3826.689941,3783.600098,3815.050049,0.0,4773040000


In [93]:
df = pd.read_csv(constants.PROCESSED_DATA_DIR / "nan_summary_sp500.csv.gz")
print(df.drop(columns="Ticker").sum())
print(validation.get_nan_summary(benchmark_df))

Leading                4773
Trailing                  0
middle_Adj Close          0
middle_Close              0
middle_Dividends          0
middle_High               0
middle_Low                0
middle_Open               0
middle_Stock Splits       0
middle_Volume             0
dtype: int64
({'Leading': 0, 'Trailing': 0}, {'Adj Close': 0, 'Close': 0, 'Dividends': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Stock Splits': 0, 'Volume': 0})


In [94]:
benchmark_duplicate_ticker_date = benchmark_df[benchmark_df.duplicated(subset=["Date"], keep=False)]
benchmark_duplicate_ticker_date

,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume


Volume and price check - no non positive price or negaive volumes.

In [95]:
price_cols = ["Open", "High", "Low", "Close", "Adj Close"]
df = sp500_df.dropna(subset=["Close"])

print("Nonpositive price:", (df[price_cols] <= 0).any(axis=1).sum())
print("Negative volume:", (df["Volume"] < 0).sum())
print("Zero volume:", (df["Volume"] == 0).sum())
print("High below Low:", (df["High"] < df["Low"]).sum())

Nonpositive price: 0
Negative volume: 0
Zero volume: 451
High below Low: 0


In [96]:
splits = df.loc[df["Stock Splits"] > 0, ["Ticker", "Date", "Stock Splits"]]
print(f"{len(splits)} splits ({splits['Ticker'].nunique()} tickers)")
splits

61 splits (53 tickers)


,Ticker,Date,Stock Splits
31513,AMZN,2022-06-06,20.00
32382,ANET,2021-11-18,4.00
33146,ANET,2024-12-04,4.00
37227,APH,2021-03-05,2.00
38050,APH,2024-06-12,2.00
...,...,...,...
491230,WMT,2024-02-26,3.00
491753,WRB,2022-03-24,1.50
492329,WRB,2024-07-11,1.50
493332,WSM,2024-07-09,2.00


In [97]:
zero_volume = df[df["Volume"] == 0]
print(zero_volume["Ticker"].value_counts())
print(zero_volume["Date"].value_counts().head(10))

Ticker
SW      424
FERG     27
Name: count, dtype: int64
Date
2021-01-12    2
2021-01-13    2
2021-01-20    2
2021-01-21    2
2021-01-22    2
2021-01-25    2
2021-01-26    2
2021-01-28    2
2021-02-02    2
2021-02-03    2
Name: count, dtype: int64


Smurfit Westrock (SW) listed after a merger and Ferguson (Ferg) are outside the RFM window anyway

In [98]:
for t in ["SW", "FERG"]:
    s = df[df["Ticker"] == t]
    print(
        t,
        ", last zero-volume day:",
        s.loc[s["Volume"] == 0, "Date"].max().date(),
        ", first non-zero day:",
        s.loc[s["Volume"] > 0, "Date"].min().date(),
    )

SW , last zero-volume day: 2024-07-05 , first non-zero day: 2021-01-04
FERG , last zero-volume day: 2021-03-05 , first non-zero day: 2021-01-04


In [99]:
sp500_df = sp500_df[sp500_df["Ticker"] != "SW"]
ferg_last_zero = df.loc[(df["Ticker"] == "FERG") & (df["Volume"] == 0), "Date"].max()
sp500_df = sp500_df[~((sp500_df["Ticker"] == "FERG") & (sp500_df["Date"] <= ferg_last_zero))]

In [100]:
nan_save_dir = constants.PROCESSED_DATA_DIR / "nan_summary_benchmark.yaml"

with open(nan_save_dir, "w") as f:
    safe_dump(validation.get_nan_summary(benchmark_df), f, sort_keys=False)
    print(f"Saved benchmark NaN summary to {nan_save_dir}")

Saved benchmark NaN summary to D:\dev\crm-rfm-sp500\data\processed\nan_summary_benchmark.yaml


In [101]:
exclusion_file_path = (
    constants.PROCESSED_DATA_DIR / f"exclusion_dict_{start_date}_{end_date_exclusive}.yaml"
)

with open(exclusion_file_path) as f:
    exclusions = safe_load(f)

exclusions["SW"] = "Only 6 months of valid data, as NYSE listed on 07/08/2024"

with open(exclusion_file_path, "w") as f:
    safe_dump(exclusions, f, sort_keys=False)

print(exclusions)

{'FDXF': 'No data available', 'HONA': 'No data available', 'Q': 'No data available', 'SNDK': 'No data available', 'VYLR': 'No data available', 'SW': 'Only 6 months of valid data, as NYSE listed on 07/08/2024', 'GEV': 'Only 39 weeks of returns which is below the 52 minimum for beta', 'RDDT': 'Only 40 weeks of returns which is below the 52 minimum for beta', 'SOLV': 'Only 39 weeks of returns which is below the 52 minimum for beta'}


Group days into weeks that end of Friday or use Thursday if Firday was a holiday

In [102]:
prices = sp500_df.pivot(index="Date", columns="Ticker", values="Adj Close")
weekly_prices = prices.resample("W-FRI").last()

bench = benchmark_df.set_index("Date")["Adj Close"]
weekly_bench = bench.resample("W-FRI").last()

weekly_prices.tail()

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-06,138.410217,241.090378,165.290604,136.919998,110.698914,97.830002,347.586212,552.960022,211.860138,48.489246,...,29.497431,92.786140,65.464279,107.394981,123.499733,98.250000,133.071320,105.882988,409.320007,171.459137
2024-12-13,136.854584,246.342300,162.645035,130.949997,108.573029,93.419998,343.733704,465.690002,211.206848,49.413399,...,28.337893,92.943451,63.943611,104.813416,117.860786,91.980003,132.462173,104.618622,403.589996,173.130371
2024-12-20,132.433884,252.656464,164.718338,134.210007,109.386993,90.989998,351.103271,447.170013,206.497314,47.612255,...,25.924555,87.457092,63.267750,100.113655,114.479370,89.650002,127.975883,104.991066,393.040009,160.168427
2024-12-27,133.447983,253.748535,166.997986,133.384995,110.114761,92.339996,341.337860,446.480011,211.577362,47.697124,...,26.386482,87.191635,63.962376,100.690475,114.860504,88.970001,130.750809,104.746033,389.070007,159.935242
2025-01-03,132.510971,248.615784,166.707169,131.410004,108.314491,92.349998,337.130829,444.679993,207.160355,47.640545,...,26.537315,84.713936,63.380386,101.721214,113.384804,84.989998,129.716278,103.763702,386.220001,158.312531


Remove incomplete week

In [103]:
weekly_prices = weekly_prices.loc[:"2024-12-27"]
weekly_bench = weekly_bench.loc[:"2024-12-27"]

print(weekly_prices.index.min().date(), "to", weekly_prices.index.max().date())

2021-01-08 to 2024-12-27


Check that benchmark and shares have the same list of weeks

In [104]:
weekly_returns = weekly_prices.pct_change(fill_method=None).iloc[1:]
bench_returns = weekly_bench.pct_change(fill_method=None).iloc[1:]

print(weekly_returns.index.equals(bench_returns.index))
print(bench_returns.isna().sum())

True
0


In [105]:
weeks_per_share = weekly_returns.notna().sum()
less_than_52 = weeks_per_share[weeks_per_share < 52]
less_than_52

Ticker
GEV     39
RDDT    40
SOLV    39
dtype: int64

In [106]:
for ticker, n in less_than_52.items():
    exclusions[ticker] = f"Only {n} weeks of returns which is below the 52 minimum for beta"

with open(exclusion_file_path, "w") as f:
    safe_dump(exclusions, f, sort_keys=False)

exclusions

{'FDXF': 'No data available',
 'HONA': 'No data available',
 'Q': 'No data available',
 'SNDK': 'No data available',
 'VYLR': 'No data available',
 'SW': 'Only 6 months of valid data, as NYSE listed on 07/08/2024',
 'GEV': 'Only 39 weeks of returns which is below the 52 minimum for beta',
 'RDDT': 'Only 40 weeks of returns which is below the 52 minimum for beta',
 'SOLV': 'Only 39 weeks of returns which is below the 52 minimum for beta'}

In [107]:
weekly_returns = weekly_returns.drop(columns=less_than_52.index)
sp500_df = sp500_df[~sp500_df["Ticker"].isin(less_than_52.index)]

temp = weekly_prices["AAPL"] / weekly_prices["AAPL"].shift(1) - 1
print((temp.iloc[1:] - weekly_returns["AAPL"]).abs().max())

print(sp500_df["Ticker"].nunique(), "shares in daily data")
print(weekly_returns.shape)

0.0
494 shares in daily data
(207, 494)


In [108]:
weekly_returns.to_csv(constants.PROCESSED_DATA_DIR / "weekly_returns.csv.gz")
bench_returns.to_csv(constants.PROCESSED_DATA_DIR / "benchmark_weekly_returns.csv.gz")
sp500_df.to_csv(constants.PROCESSED_DATA_DIR / "sp500_clean_daily.csv.gz", index=False)